# RAG — Société Générale Pilier 3 T2 2022

## C'est quoi un RAG ?

**RAG = Retrieval-Augmented Generation** (Génération Augmentée par Récupération)

Imagine que tu donnes un livre à ChatGPT et que tu lui poses des questions dessus.
Le problème : ChatGPT ne peut pas lire un livre entier d'un coup (trop grand).

La solution RAG en 3 étapes :
1. **Découper** le PDF en petits morceaux (chunks)
2. **Chercher** les morceaux qui répondent à ta question
3. **Envoyer** seulement ces morceaux à GPT pour qu'il réponde

---

### Ce dont tu as besoin
- Une clé API OpenAI (dans le fichier `.env`)
- Le fichier PDF dans le même dossier
- Exécuter les cellules dans l'ordre (Shift+Entrée)

---
## CELLULE 1 — Installation des bibliothèques

Une **bibliothèque** (ou package) = un ensemble d'outils que quelqu'un a déjà programmé pour toi.
On les installe une seule fois avec `pip`.

In [ ]:
# Le % au début veut dire que c'est une commande spéciale Jupyter (pas du Python)
# -q = quiet = silencieux (n'affiche pas tous les détails d'installation)

# langchain         = la bibliothèque principale qui orchestre tout le RAG
# langchain-openai  = la partie de langchain qui parle à OpenAI (GPT, embeddings)
# langchain-community = outils supplémentaires (chargeur PDF, FAISS...)
# langchain-text-splitters = outils pour découper le texte en morceaux
%pip install -q langchain langchain-openai langchain-community langchain-text-splitters

# faiss-cpu       = bibliothèque de Facebook pour chercher des vecteurs rapidement
# pypdf           = pour lire les fichiers PDF
# python-dotenv   = pour lire les variables secrètes depuis un fichier .env
# tiktoken        = pour compter les tokens (unités de texte utilisées par GPT)
# openai          = le SDK officiel d'OpenAI
%pip install -q faiss-cpu pypdf python-dotenv tiktoken openai

---
## CELLULE 2 — Configuration de la clé API OpenAI

La clé API = ton mot de passe pour utiliser les services OpenAI (payant au nombre de tokens).  
Elle est stockée dans un fichier `.env` pour ne pas l'écrire directement dans le code.

In [ ]:
# On importe le module 'os' = Operating System
# Il permet d'interagir avec le système (lire des variables d'environnement, etc.)
import os

# On importe la fonction load_dotenv depuis la bibliothèque python-dotenv
# Cette fonction va lire le fichier .env et charger les variables dedans
from dotenv import load_dotenv

# On appelle load_dotenv() pour lire le fichier .env
# Après cet appel, OPENAI_API_KEY sera accessible via os.getenv()
load_dotenv()

# ALTERNATIVE si tu n'as pas de fichier .env :
# Décommente la ligne suivante et remplace par ta vraie clé
# os.environ["OPENAI_API_KEY"] = "sk-proj-VOTRE_CLE_ICI"

# assert = vérifie une condition. Si la condition est fausse, Python s'arrête et affiche le message
# os.getenv("OPENAI_API_KEY") = récupère la valeur de la variable OPENAI_API_KEY
# Si la clé n'existe pas, os.getenv() renvoie None, et assert échoue
assert os.getenv("OPENAI_API_KEY"), "Clé API manquante — ajoutez-la dans .env"

# On affiche les 12 premiers caractères de la clé pour confirmer qu'elle est chargée
# (on ne montre pas toute la clé pour des raisons de sécurité)
print("Clé API chargée :", os.getenv("OPENAI_API_KEY")[:12], "...")

---
## CELLULE 3 — Chargement du PDF

On lit le PDF page par page et on transforme chaque page en un objet Python qu'on appelle `Document`.

In [ ]:
# On importe PyPDFLoader depuis langchain_community
# C'est un outil qui sait lire les fichiers PDF et extraire le texte
from langchain_community.document_loaders import PyPDFLoader

# On stocke le nom du fichier PDF dans une variable
# Le fichier doit être dans le même dossier que ce notebook
CHEMIN_PDF = "Societe-Generale-Pilier-3_T2-2022_FR.pdf"

# On crée un objet 'loader' capable de lire ce PDF
# Pour l'instant il ne lit rien encore, il est juste prêt
loader = PyPDFLoader(CHEMIN_PDF)

# On appelle .load() pour vraiment lire le PDF
# Résultat : une liste de Documents, un par page
# Chaque Document contient :
#   - .page_content : le texte de la page
#   - .metadata     : des infos sur la page (numéro de page, nom du fichier...)
pages = loader.load()

# On affiche combien de pages ont été chargées
# len() = longueur = nombre d'éléments dans la liste
print(f"PDF chargé : {len(pages)} pages")

# Affichage du texte des 200 premiers caractères de la page 1 pour vérifier
print("\nApercu page 1 :")
print(pages[0].page_content[:200])

---
## CELLULE 4 — Découpage du texte en morceaux (chunks)

GPT ne peut pas lire 136 pages d'un coup. On découpe le texte en petits morceaux de ~1000 caractères.
Le **chevauchement** (overlap) garantit qu'une phrase coupée entre deux morceaux apparaît dans les deux.

In [ ]:
# On importe RecursiveCharacterTextSplitter
# 'Recursive' = il essaie d'abord de couper aux paragraphes, puis aux lignes, puis aux phrases...
# C'est plus intelligent qu'un simple découpage à taille fixe
from langchain_text_splitters import RecursiveCharacterTextSplitter

# On configure le découpeur avec 3 paramètres :
splitter = RecursiveCharacterTextSplitter(
    
    # chunk_size = taille maximale d'un morceau en nombre de caractères
    # 1000 caractères ≈ environ 150-200 mots ≈ 250 tokens
    chunk_size=1000,
    
    # chunk_overlap = combien de caractères sont partagés entre deux morceaux consécutifs
    # Exemple : si le morceau 1 finit à la lettre 1000, le morceau 2 commence à la lettre 800
    # Cela évite de perdre du contexte aux jonctions
    chunk_overlap=200,
    
    # separators = la liste des caractères où couper, dans l'ordre de priorité
    # On coupe d'abord aux doubles retours à la ligne (paragraphes)
    # puis aux simples retours à la ligne, puis aux points, puis aux espaces, etc.
    separators=["\n\n", "\n", ". ", " ", ""]
)

# On applique le découpage sur notre liste de pages
# Résultat : une nouvelle liste de Documents (les morceaux)
# split_documents() conserve les métadonnées (numéro de page) dans chaque morceau
chunks = splitter.split_documents(pages)

# Affichage du nombre total de morceaux créés
print(f"Nombre de morceaux (chunks) : {len(chunks)}")

# On affiche le contenu du 10ème morceau (index 10) pour vérifier
# [10] = le 11ème élément (les listes commencent à 0 en Python)
print("\n--- Exemple : chunk numero 10 ---")
print(f"Page source : {chunks[10].metadata.get('page', '?') + 1}")
print(f"Texte : {chunks[10].page_content[:400]}")

---
## CELLULE 5 — Création de l'index vectoriel FAISS

### C'est quoi un vecteur / embedding ?
Un **embedding** transforme du texte en une liste de nombres (ex: [0.2, -0.5, 0.8, ...]).  
Des textes qui parlent du **même sujet** donnent des vecteurs **proches** en espace mathématique.  
**FAISS** (de Facebook) permet de chercher très rapidement quel vecteur est le plus proche d'un autre.

In [ ]:
# On importe OpenAIEmbeddings = le modèle d'OpenAI qui convertit du texte en vecteurs
from langchain_openai import OpenAIEmbeddings

# On importe FAISS = la base de données de vecteurs de Facebook/Meta
from langchain_community.vectorstores import FAISS

# On crée le modèle d'embeddings en précisant quel modèle OpenAI utiliser
# "text-embedding-3-small" = modèle récent, rapide et pas cher
# Il transforme n'importe quel texte en un vecteur de 1536 dimensions
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# Message d'information pour patienter (la vectorisation appelle l'API OpenAI)
print("Vectorisation en cours... (10-30 secondes selon la connexion)")
print("Chaque chunk est envoyé à OpenAI qui renvoie un vecteur de nombres.")

# FAISS.from_documents() fait 3 choses en une :
# 1. Envoie chaque chunk à OpenAI → reçoit son vecteur
# 2. Stocke tous les vecteurs dans une structure FAISS
# 3. Retourne l'objet vectorstore qu'on pourra interroger
vectorstore = FAISS.from_documents(chunks, embeddings)

# vectorstore.index.ntotal = nombre total de vecteurs stockés dans FAISS
# Ce nombre doit être égal au nombre de chunks
print(f"\nIndex FAISS cree : {vectorstore.index.ntotal} vecteurs")
print("Chaque chunk du PDF a maintenant son propre vecteur numerique.")

In [ ]:
# On sauvegarde l'index FAISS sur le disque dur
# Ainsi, si on relance le notebook plus tard, on n'aura pas à tout re-vectoriser
# (la vectorisation coûte de l'argent et du temps)
# Un dossier 'faiss_index_sg_pilier3' sera créé dans le répertoire courant
vectorstore.save_local("faiss_index_sg_pilier3")

print("Index sauvegarde sur le disque dans : ./faiss_index_sg_pilier3/")
print("La prochaine fois, tu pourras le recharger sans re-vectoriser (voir Cellule 9).")

---
## CELLULE 6 — Construction de la chaîne RAG

On assemble maintenant toutes les pièces ensemble avec **LCEL** (LangChain Expression Language).  
La syntaxe `A | B | C` signifie : "la sortie de A va dans B, la sortie de B va dans C".

```
Question de l'utilisateur
        ↓
   [RETRIEVER] → cherche les 4 chunks les plus similaires dans FAISS
        ↓
   [PROMPT] → construit le message : contexte + question
        ↓
   [LLM GPT] → génère la réponse
        ↓
   [PARSER] → extrait le texte de la réponse
        ↓
   Réponse finale
```

In [ ]:
# ChatOpenAI = la classe pour utiliser GPT-4 ou GPT-4o-mini d'OpenAI
from langchain_openai import ChatOpenAI

# ChatPromptTemplate = permet de créer un modèle de message avec des variables
# Les variables sont entre accolades : {context} et {question}
from langchain_core.prompts import ChatPromptTemplate

# StrOutputParser = transforme la réponse du modèle en simple texte Python (string)
# Sans lui, la réponse serait un objet complexe LangChain
from langchain_core.output_parsers import StrOutputParser

# RunnablePassthrough = composant qui laisse passer la donnée sans la modifier
# On l'utilise pour "passer" la question telle quelle dans la chaîne
from langchain_core.runnables import RunnablePassthrough


# --- ÉTAPE 1 : Créer le retriever ---
# .as_retriever() transforme notre index FAISS en un outil de recherche
# search_kwargs={"k": 4} = on veut les 4 chunks les plus similaires à la question
# (k=4 est un bon compromis entre contexte suffisant et ne pas surcharger GPT)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})


# --- ÉTAPE 2 : Créer le prompt (le message envoyé à GPT) ---
# On définit un template de message avec 2 variables :
#   {context} = les chunks retrouvés par FAISS
#   {question} = la question de l'utilisateur
# Le triple guillemet """ permet d'écrire sur plusieurs lignes
PROMPT_TEMPLATE = """
Tu es un analyste financier expert. Reponds a la question en te basant UNIQUEMENT
sur le contexte extrait du rapport Pilier 3 T2 2022 de la Societe Generale ci-dessous.
Si l'information n'est pas dans le contexte, dis-le clairement.
Cite les pages sources quand c'est possible.

CONTEXTE :
{context}

QUESTION : {question}

REPONSE :
"""

# .from_template() crée un objet prompt à partir de notre texte
# Les variables {context} et {question} seront remplies automatiquement
prompt = ChatPromptTemplate.from_template(PROMPT_TEMPLATE)


# --- ÉTAPE 3 : Choisir le modèle LLM ---
# ChatOpenAI = interface pour discuter avec GPT
llm = ChatOpenAI(
    # Le modèle à utiliser :
    # "gpt-4o-mini" = rapide et pas cher (recommandé pour commencer)
    # "gpt-4o"      = plus intelligent mais plus coûteux
    model="gpt-4o-mini",
    
    # temperature = créativité du modèle (0 = très factuel, 1 = créatif)
    # Pour un document financier, on veut 0 : pas d'inventions !
    temperature=0
)


# --- ÉTAPE 4 : Fonction de formatage des chunks ---
# Cette fonction reçoit une liste de Documents (chunks)
# et les colle ensemble en un seul texte avec le numéro de page
def formater_docs(docs):
    # "\n\n".join([...]) = colle les éléments avec 2 sauts de ligne entre chaque
    return "\n\n".join(
        # Pour chaque document 'd' dans la liste 'docs' :
        # - d.metadata.get('page', '?') = récupère le numéro de page (0-indexé)
        # - +1 = pour afficher les pages à partir de 1 (plus naturel)
        # - d.page_content = le texte du chunk
        f"[Page {d.metadata.get('page', '?')+1}] {d.page_content}"
        for d in docs  # boucle sur chaque chunk
    )


# --- ÉTAPE 5 : Assembler la chaîne RAG avec LCEL ---
# La syntaxe | est comme un tuyau : les données coulent de gauche à droite
chaine_rag = (
    # On crée un dictionnaire avec 2 clés :
    {
        # "context" : on prend la question → retriever cherche les chunks → formater_docs les formate
        # Le | ici signifie : passe la question dans le retriever, puis le résultat dans formater_docs
        "context": retriever | formater_docs,
        
        # "question" : on laisse passer la question sans la modifier (RunnablePassthrough)
        "question": RunnablePassthrough()
    }
    # | prompt : on passe le dictionnaire {context, question} dans le template de prompt
    | prompt
    
    # | llm : on envoie le prompt rempli au modèle GPT
    | llm
    
    # | StrOutputParser() : on extrait le texte de la réponse GPT (enlève la structure LangChain)
    | StrOutputParser()
)

print("Chaine RAG assemblée et prete !")
print("Question → FAISS → Contexte → GPT → Reponse")

---
## CELLULE 7 — Fonction utilitaire pour poser des questions

In [ ]:
# On crée une fonction Python réutilisable pour poser une question
# def = mot-clé pour définir une fonction
# poser_question = nom de la fonction (on choisit ce nom)
# question: str = le paramètre attendu (doit être du texte)
# afficher_sources: bool = paramètre optionnel (True par défaut)
def poser_question(question: str, afficher_sources: bool = True):
    
    # Affiche une ligne de séparation pour la lisibilité
    # '='*70 = répète le caractère '=' 70 fois
    print(f"\n{'='*70}")
    
    # Affiche la question posée
    print(f"QUESTION : {question}")
    print('='*70)
    
    # .invoke() = lance la chaîne RAG avec la question
    # C'est ici que tout se passe :
    # 1. FAISS cherche les 4 chunks les plus pertinents
    # 2. Le prompt est rempli avec ces chunks + la question
    # 3. GPT génère une réponse
    # 4. Le parser extrait le texte
    reponse = chaine_rag.invoke(question)
    
    # Affiche la réponse de GPT
    print(f"\nREPONSE :\n{reponse}")
    
    # Si l'utilisateur veut voir les sources (afficher_sources=True par défaut)
    if afficher_sources:
        
        # On refait une recherche pour savoir quels chunks ont été utilisés
        # retriever.invoke() renvoie la liste des 4 Documents les plus pertinents
        docs_sources = retriever.invoke(question)
        
        # On extrait les numéros de page de chaque document
        # set() = élimine les doublons (si 2 chunks viennent de la même page)
        # sorted() = trie par ordre croissant
        # d.metadata.get('page', 0) = numéro de page (0-indexé), +1 pour afficher à partir de 1
        pages_sources = sorted(set(d.metadata.get('page', 0)+1 for d in docs_sources))
        
        # Affiche les pages utilisées comme source
        print(f"\nSOURCES (pages du PDF) : {pages_sources}")

---
## CELLULE 8 — Questions sur le rapport Société Générale

Exécute chaque cellule ci-dessous pour interroger le rapport Pilier 3 !

In [ ]:
# Question 1 : Le ratio CET1 est un indicateur clé de solidité financière
# CET1 = Common Equity Tier 1 = fonds propres de base de la banque
poser_question("Quelle est la date de publication du document RAPPORT sur les Risques?")

In [ ]:
# Question 2 : Les RWA (Risk-Weighted Assets) mesurent les risques pondérés
# Plus les RWA sont élevés, plus la banque doit avoir de fonds propres
poser_question("Quel est le montant total des actifs ponderes par les risques (RWA) ?")

In [ ]:
# Question 3 : La liquidité = capacité à faire face aux retraits et paiements urgents
# LCR = Liquidity Coverage Ratio (ratio à court terme, minimum 100% réglementaire)
# NSFR = Net Stable Funding Ratio (ratio à long terme, minimum 100% réglementaire)
poser_question("Quelle est la situation de liquidite du groupe (LCR, NSFR) ?")

In [ ]:
# Question 4 : Le risque de crédit = risque que les emprunteurs ne remboursent pas
poser_question("Comment evolue le risque de credit de la Societe Generale ?")

In [ ]:
# Question 5 : Vue d'ensemble des risques du document
poser_question("Quels sont les principaux risques identifies dans ce rapport Pilier 3 ?")

In [ ]:
# Question 6 : Posez VOTRE propre question ici !
# Changez simplement le texte entre les guillemets
ma_question = "Quelle est l'exposition au risque de marche de la banque ?"

# On appelle la fonction avec notre question
poser_question(ma_question)

---
## CELLULE 9 — Rechargement rapide (si tu relances le notebook)

Si tu fermes Jupyter et tu le relances, toutes les variables sont perdues.  
Mais l'index FAISS est sauvegardé sur le disque → **pas besoin de re-vectoriser** (économie de temps et d'argent).  
Exécute juste cette cellule au lieu des cellules 3, 4 et 5.

In [ ]:
# --- Imports nécessaires ---
# (identiques aux cellules 2, 3, 5 et 6 : on repart de zéro car les variables
#  Python sont perdues à chaque redémarrage du kernel/notebook)
import os
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# --- Clé API ---
# On relit le fichier .env pour remettre OPENAI_API_KEY dans les variables d'environnement
load_dotenv()

# --- Rechargement de l'index FAISS depuis le disque ---
# On recrée le même modèle d'embedding que celui utilisé à la création de l'index
# (obligatoire : les vecteurs stockés ne veulent dire quelque chose que pour CE modèle précis)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# FAISS.load_local() = lit les fichiers index.faiss + index.pkl et reconstruit l'objet vectorstore
# 1er argument  = nom du dossier où l'index a été sauvegardé (cellule 5)
# 2e argument   = le modèle d'embedding (nécessaire pour vectoriser les futures questions)
# allow_dangerous_deserialization=True = obligatoire pour charger un index FAISS local
#   (mesure de sécurité de langchain : on confirme qu'on fait confiance au fichier,
#    ici sans risque puisque c'est nous-mêmes qui l'avons créé à la cellule 5)
vectorstore = FAISS.load_local(
    "faiss_index_sg_pilier3",             # nom du dossier où l'index est sauvegardé
    embeddings,                            # le modèle d'embedding (doit être identique à la création)
    allow_dangerous_deserialization=True   # confirmation de sécurité
)

# On vérifie que le nombre de vecteurs rechargés correspond bien à ce qui a été sauvegardé
print(f"Index rechargé : {vectorstore.index.ntotal} vecteurs")

# --- Reconstruction du retriever ---
# On recrée le retriever qui cherche les k=4 chunks les plus proches d'une question
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

# --- Reconstruction de la chaîne RAG ---
# On recrée exactement le même template de prompt que dans la cellule 6
# {context} sera rempli par les chunks trouvés, {question} par la question de l'utilisateur
PROMPT_TEMPLATE = """
Tu es un analyste financier expert. Reponds a la question en te basant UNIQUEMENT
sur le contexte extrait du rapport Pilier 3 T2 2022 de la Societe Generale ci-dessous.
Si l'information n'est pas dans le contexte, dis-le clairement.
Cite les pages sources quand c'est possible.

CONTEXTE :
{context}

QUESTION : {question}

REPONSE :
"""
# On transforme le texte en objet prompt utilisable par la chaîne LCEL
prompt = ChatPromptTemplate.from_template(PROMPT_TEMPLATE)

# Modèle GPT : gpt-4o-mini (rapide/pas cher), temperature=0 (factuel, pas d'invention)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Fonction de formatage : colle les chunks retrouvés en un seul texte,
# en préfixant chacun par son numéro de page (utile pour que GPT cite ses sources)
def formater_docs(docs):
    return "

".join(
        f"[Page {d.metadata.get('page','?')+1}] {d.page_content}"
        for d in docs
    )

# Chaîne RAG (LCEL) : question -> {context, question} -> prompt -> llm -> texte brut
# Identique à la cellule 6 : retriever cherche, formater_docs formate, prompt assemble,
# llm génère, StrOutputParser() extrait le texte final
chaine_rag = (
    {"context": retriever | formater_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# Fonction pour poser des questions (identique à la cellule 7)
# question         = la question posée par l'utilisateur
# afficher_sources = si True, affiche aussi les numéros de page utilisés comme sources
def poser_question(question: str, afficher_sources: bool = True):
    # Ligne de séparation + rappel de la question posée
    print(f"
{'='*70}
QUESTION : {question}
{'='*70}")
    # Lance toute la chaîne RAG (recherche FAISS -> prompt -> GPT -> texte)
    reponse = chaine_rag.invoke(question)
    print(f"
REPONSE :
{reponse}")
    if afficher_sources:
        # Recherche séparée pour connaître les chunks utilisés et donc leurs pages
        docs_sources = retriever.invoke(question)
        # set() enlève les doublons, sorted() trie par numéro de page croissant
        pages_sources = sorted(set(d.metadata.get('page', 0)+1 for d in docs_sources))
        print(f"
SOURCES (pages) : {pages_sources}")

print("Chaine RAG rechargee et prete. Tu peux poser tes questions !")